In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)
from datetime import datetime

cdc_data = [
    # C000001 - two attributes changed at the same timestamp
    ("C000001", "phone", "1111111111", "2222222222", datetime(2026, 8, 20, 10, 0, 0)),
    ("C000001", "email", "C000001_old@gmail.com", "C000001_new@gmail.com", datetime(2026, 8, 20, 10, 0, 0)),

    # C000001 - another change later
    ("C000001", "phone", "2222222222", "3333333333", datetime(2026, 8, 20, 14, 0, 0)),

    # C000002 - customer type changed
    ("C000002", "customer_type", "REGULAR", "PREMIUM", datetime(2026, 8, 21, 9, 30, 0)),

    # C000002 - email changed later
    ("C000002", "email", "C000002_old@gmail.com", "C000002_new@gmail.com", datetime(2026, 8, 21, 15, 0, 0)),

    # C000003 - phone changed
    ("C000003", "phone", "4444444444", "5555555555", datetime(2026, 8, 22, 11, 0, 0)),
]

schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("old_value", StringType(), True),
    StructField("new_value", StringType(), True),
    StructField("event_timestamp", TimestampType(), False)
])

customer_cdc_df = spark.createDataFrame(cdc_data, schema)

display(customer_cdc_df.orderBy("customer_id", "event_timestamp"))

In [0]:
customer_cdc_table = "dbx_fintech_data_platform.bronze.customer_cdc"

(
    customer_cdc_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(customer_cdc_table)
)

In [0]:
display(
    spark.table("dbx_fintech_data_platform.bronze.customer_cdc")
    .orderBy("customer_id", "event_timestamp")
)

In [0]:
df_customer_cdc = spark.table("dbx_fintech_data_platform.bronze.customer_cdc").orderBy("customer_id", "event_timestamp")
df_customer_cdc.display()

In [0]:
df_customer_cdc_pivot = (
    df_customer_cdc.groupBy("customer_id","event_timestamp")
    .pivot("column_name",["phone","email","customer_type"])
    .agg(
        F.first("old_value").alias("old_value"),
        F.first("new_value").alias("new_value"))
)

display(df_customer_cdc_pivot)

In [0]:
win = Window.partitionBy("customer_id").orderBy("event_timestamp").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_pivot_complete = (
    df_customer_cdc_pivot.withColumn("email_old_value", F.last_value("email_old_value",True).over(win))
    .withColumn("email_new_value", F.last_value("email_new_value",True).over(win))
    .withColumn("phone_old_value", F.last_value("phone_old_value",True).over(win))
    .withColumn("phone_new_value", F.last_value("phone_new_value",True).over(win))
    .withColumn("customer_type_old_value", F.last_value("customer_type_old_value",True).over(win))
    .withColumn("customer_type_new_value", F.last_value("customer_type_new_value",True).over(win))
)
df_pivot_complete.orderBy("customer_id", "event_timestamp").display()

In [0]:
df_customer_silver = (
    spark.table("dbx_fintech_data_platform.silver.customers")
)

In [0]:
df_customer_silver.display()

In [0]:
df_pivot_complete = (
    df_pivot_complete.join(df_customer_silver, on = "customer_id", how="left")
    .withColumn("customer_type_new_value", F.when(df_pivot_complete.customer_type_new_value.isNull(), df_customer_silver.customer_type).otherwise(df_pivot_complete.customer_type_new_value))
    .withColumn("email_new_value", F.when(df_pivot_complete.email_new_value.isNull(), df_customer_silver.email).otherwise(df_pivot_complete.email_new_value))
    .withColumn("phone_new_value", F.when(df_pivot_complete.phone_new_value.isNull(), df_customer_silver.phone).otherwise(df_pivot_complete.phone_new_value))
    .drop(*[c for c in df_customer_silver.columns if c != "customer_id"])
)

df_pivot_complete.orderBy("customer_id","event_timestamp").display()

In [0]:
df_pivot_complete = (
    df_pivot_complete
    .withColumn("effective_from", F.col("event_timestamp"))
    .withColumn("effective_to", F.lead("event_timestamp", 1).over(Window.partitionBy("customer_id").orderBy("event_timestamp")))
    .withColumn("is_current", F.when(F.col("effective_to").isNull(), True).otherwise(False))
)
df_pivot_complete.orderBy("customer_id","event_timestamp").display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

attr_win = (
    Window
    .partitionBy("customer_id", "column_name")
    .orderBy("event_timestamp")
)

df_initial_values = (
    df_customer_cdc
    .withColumn(
        "rn",
        F.row_number().over(attr_win)
    )
    .filter(F.col("rn") == 1)
    .select(
        "customer_id",
        "column_name",
        F.col("old_value").alias("initial_value")
    )
)

display(df_initial_values)

In [0]:
df_initial_pivot = (
    df_initial_values
    .groupBy("customer_id")
    .pivot("column_name", ["phone", "email", "customer_type"])
    .agg(F.first("initial_value"))
    .withColumnRenamed("phone", "initial_phone")
    .withColumnRenamed("email", "initial_email")
    .withColumnRenamed("customer_type", "initial_customer_type")
)

display(df_initial_pivot)

In [0]:
df_pivot_complete = (
    df_pivot_complete
    .join(
        df_initial_pivot,
        on="customer_id",
        how="left"
    )
)

In [0]:
df_pivot_complete = (
    df_pivot_complete
    .withColumn(
        "customer_type",
        F.coalesce(
            F.col("customer_type_new_value"),
            F.col("initial_customer_type")
        )
    )
    .withColumn(
        "email",
        F.coalesce(
            F.col("email_new_value"),
            F.col("initial_email")
        )
    )
    .withColumn(
        "phone",
        F.coalesce(
            F.col("phone_new_value"),
            F.col("initial_phone")
        )
    )
)
df_pivot_complete.display()

In [0]:
df_pivot_complete.select(
    "customer_id",
    "event_timestamp",
    "phone",
    "email",
    "customer_type",
    "effective_from",
    "effective_to",
    "is_current"
).orderBy(
    "customer_id",
    "event_timestamp"
).display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# First CDC event for each customer
first_cdc_df = (
    df_customer_cdc
    .groupBy("customer_id")
    .agg(
        F.min("event_timestamp").alias("first_event_timestamp")
    )
)

display(first_cdc_df)

In [0]:
df_initial_scd2 = (
    df_customer_silver
    .join(
        first_cdc_df,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "effective_from",
        F.col("created_at")
    )
    .withColumn(
        "effective_to",
        F.col("first_event_timestamp")
    )
    .withColumn(
        "is_current",
        F.when(
            F.col("first_event_timestamp").isNull(),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .drop("first_event_timestamp")
)

In [0]:
display(
    df_initial_scd2
    .select(
        "customer_id",
        "customer_type",
        "phone",
        "email",
        "created_at",
        "effective_from",
        "effective_to",
        "is_current"
    )
    .orderBy("customer_id")
)

In [0]:
df_cdc_scd2 = (
    df_pivot_complete
    .join(
        df_customer_silver.select(
            "customer_id",
            "created_at",
            "updated_at"
        ),
        on="customer_id",
        how="left"
    )
    .select(
        "customer_id",
        F.col("customer_type_new_value").alias("customer_type"),
        F.col("phone_new_value").alias("phone"),
        F.col("email_new_value").alias("email"),
        "created_at",
        "updated_at",
        "effective_from",
        "effective_to",
        "is_current"
    )
)

display(
    df_cdc_scd2
    .orderBy("customer_id", "effective_from")
)

In [0]:
df_customer_scd2 = (
    df_initial_scd2
    .select(
        "customer_id",
        "customer_type",
        "phone",
        "email",
        "created_at",
        "updated_at",
        "effective_from",
        "effective_to",
        "is_current"
    )
    .unionByName(
        df_cdc_scd2.select(
            "customer_id",
            "customer_type",
            "phone",
            "email",
            "created_at",
            "updated_at",
            "effective_from",
            "effective_to",
            "is_current"
        )
    )
)

In [0]:
display(
    df_customer_scd2
    .orderBy("customer_id", "effective_from")
)

In [0]:
display(
    df_customer_scd2
    .groupBy("customer_id")
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_count"),
        F.count("*").alias("version_count")
    )
    .orderBy("customer_id")
)

In [0]:
overlap_win = (
    Window
    .partitionBy("customer_id")
    .orderBy("effective_from")
)

df_overlap_check = (
    df_customer_scd2
    .withColumn(
        "previous_effective_to",
        F.lag("effective_to").over(overlap_win)
    )
    .filter(
        F.col("previous_effective_to").isNotNull() &
        (F.col("effective_from") < F.col("previous_effective_to"))
    )
)

display(df_overlap_check)

In [0]:
customer_scd2_table = "dbx_fintech_data_platform.silver.customers_scd2"

(
    df_customer_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(customer_scd2_table)
)

In [0]:
df_customer_scd2_target = spark.table(
    "dbx_fintech_data_platform.silver.customers_scd2"
)

display(
    df_customer_scd2_target
    .orderBy("customer_id", "effective_from")
)

In [0]:
display(
    df_customer_scd2_target
    .groupBy("customer_id")
    .agg(
        F.sum(
            F.when(F.col("is_current"), 1).otherwise(0)
        ).alias("current_count")
    )
    .filter(F.col("current_count") != 1)
)

In [0]:
overlap_win = (
    Window
    .partitionBy("customer_id")
    .orderBy("effective_from")
)

df_overlap = (
    df_customer_scd2_target
    .withColumn(
        "previous_effective_to",
        F.lag("effective_to").over(overlap_win)
    )
    .filter(
        F.col("previous_effective_to").isNotNull() &
        (F.col("effective_from") < F.col("previous_effective_to"))
    )
)

display(df_overlap)

In [0]:
display(
    df_customer_scd2_target
    .filter(
        F.col("is_current") &
        F.col("effective_to").isNotNull()
    )
)

In [0]:
display(
    df_customer_scd2_target
    .filter(
        (~F.col("is_current")) &
        F.col("effective_to").isNull()
    )
)